# Customer Churn Prediction

In [ ]:
# Importing Dependencies
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Loading dataset
df = pd.read_excel("/config/workspace/Dataset/customer_churn_large_dataset.xlsx")

In [ ]:
df.head()

In [ ]:
df.to_csv("/config/workspace/Dataset/customer_churn_large_dataset_csv.csv")

In [ ]:
df = pd.read_csv("/config/workspace/Dataset/customer_churn_large_dataset_csv.csv")

In [ ]:
df.head()

In [ ]:
df.drop(labels=("Unnamed: 0"),axis = 1, inplace=True)

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
# Data Cleaning

In [ ]:
df.isnull().sum()

- There is no missing value

In [ ]:
# Dropping the unwanted column

df.drop(labels = ['CustomerID', 'Name'], axis = 1, inplace = True)

- CustomerID and Name feature is not related to churn so we can drop these unwanted column

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df.head()

In [ ]:
# categories in Location
df.Location.value_counts()

In [ ]:
# Box Plot
sns.boxplot(df.Monthly_Bill)

In [ ]:
sns.boxplot(df.Total_Usage_GB)

- From above boxplot it's clear that there is no outlier

In [ ]:
df.Churn.value_counts()

- Dataset is balance

In [ ]:
df.head()

In [ ]:
# Splitting into Independent and dependent features

x = df.drop(labels=('Churn'),axis = 1)
y = df.Churn

In [ ]:
x

In [ ]:
y

In [ ]:
# Train test split

from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test = train_test_split(x,y,
                                                 test_size=0.40,
                                                 random_state=42)

In [ ]:
x_train

In [ ]:
from sklearn.impute import SimpleImputer # handing missing value
from sklearn.preprocessing import OneHotEncoder #handling categorical value
from sklearn.preprocessing import StandardScaler # Feature scaling

from sklearn.pipeline import Pipeline #automate the entire
from sklearn.compose import ColumnTransformer

In [ ]:
x_test

In [ ]:
df.columns

In [ ]:
categorical_cols = ['Gender', 'Location']
numerical_cols = ['Age', 'Subscription_Length_Months',
       'Monthly_Bill', 'Total_Usage_GB']

In [ ]:
# feature Engineering automation
# numerical pipeline
num_pipeline = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='median')), #missing value
        ('scaler',StandardScaler()) # feature scaling
    ]
)

# categorical pipeline
cat_pipeline = Pipeline(
    steps=[
        ("imputer",SimpleImputer(strategy='most_frequent')), #handling missing value
        ('onehotencoder',OneHotEncoder()) #encoding
    ]

)

In [ ]:
preprocessor = ColumnTransformer([
    ('num_pipeline',num_pipeline,numerical_cols),
    ('cat_pipeline',cat_pipeline,categorical_cols)
])

In [ ]:
x_train = preprocessor.fit_transform(x_train)
x_test = preprocessor.transform(x_test)

In [ ]:
x_train

In [ ]:
x_test

In [ ]:
# pickling the preprocessor
import pickle
file = open('/config/workspace/Model/preprocessor.pkl','wb')
pickle.dump(preprocessor,file)
file.close()

In [ ]:
# Model training

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
def evaluate_model(x_train,y_train,x_test,y_test,models):
    
    report = {}
    
    for i in range(len(models)):
        model = list(models.values())[i]
        
        # train model
        model.fit(x_train,y_train)
        
        #predict testing data
        y_test_pred = model.predict(x_test)
        
        # get accuracy for test data prediction
        test_model_score = accuracy_score(y_test,y_test_pred)
        
        report[list(models.keys())[i]] = test_model_score
        
    return report

In [ ]:
models = {
    'Random Forest': RandomForestClassifier(),
    'Decision Tree': DecisionTreeClassifier(),
    'Logistic Regression' : LogisticRegression(),
    'Naive Bayes': GaussianNB()
}

In [ ]:
evaluate_model(x_train,y_train,x_test,y_test,models)

In [ ]:
# hyperparameter tunnig
classifier = RandomForestClassifier()

params = {'max_depth':[3,5,10,None],
          'n_estimators':[100,200,300],
          'criterion': ['gini','entropy','log_loss'],
         }

In [ ]:
from sklearn.model_selection import GridSearchCV

clf = GridSearchCV(classifier, param_grid=params,scoring='accuracy',cv=5,verbose=3)

In [ ]:
clf.fit(x_train,y_train)

In [ ]:
clf.best_params_

In [ ]:
y_pred = clf.predict(x_test)
accuracy_score(y_test,y_pred)

In [ ]:
# Final Model
model = RandomForestClassifier(criterion='entropy', max_depth= 3, n_estimators=300)
model.fit(x_train,y_train)

In [ ]:
# pickling the model
file = open('/config/workspace/Model/model.pkl','wb')
pickle.dump(model,file)
file.close()